# 18 · Latent feature compression with SVD / Compresión de características latentes con SVD

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/18-feature-compression.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#6366f1,rgba(99,102,241,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#6366f1">DEEP DIVE · TAKE-HOME · ABOUT 40 MINUTES / ESTUDIO A FONDO · PARA DESPUÉS · UNOS 40 MINUTOS</span>

Unfold a synthetic image into a matrix, retain its dominant patterns and reconstruct the original axes. Measure energy and factor storage separately; inspect what the numbers miss in the image.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Matriciza una imagen sintética, conserva sus patrones dominantes y reconstruye los ejes originales. Mide por separado energía y almacenamiento de factores; observa qué detalles omiten las cifras.</div></div>

## What you will be able to do / Lo que podrás hacer

- Implement rank-k reconstruction of an HWC tensor using a documented unfolding.
- Test the energy/error identity and find the smallest rank meeting a threshold.
- Compare ranks 1, 5, 20 and 50, storage cost and a CP reconstruction.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0">Implementar reconstrucción de rango k con una matricización documentada.</li><li style="margin:.35em 0">Comprobar la identidad energía/error y encontrar el menor rango para un umbral.</li><li style="margin:.35em 0">Comparar rangos 1, 5, 20 y 50, coste de almacenamiento y reconstrucción CP.</li></ul></div>

<!-- CORE-PATH -->
**Core path · about 40 minutes.** Read the unfolding choice, run Setup, implement
the starter, then test before revealing the folded solution. Compare the four
ranks and check the energy curve. **Explore later:** rank slider, GIFs and CP.

**Two outcomes:** reconstruct an HWC tensor at matrix rank `k`; distinguish
retained energy, storage cost and visual fidelity.

*Ruta esencial: elige una matricización, reconstruye el tensor y comprueba
energía y error. Después explora rangos y la diferencia con CP.*

## Problem: what does it mean to compress a feature map?
*Problema: conservar patrones dominantes con menos factores.*

A CNN may emit a feature tensor `(Height, Width, Channels)`. We choose one
explicit unfolding: `(H, W, C) → (H, W * C)`, with column `w * C + c` identifying
a spatial column and channel. A shared basis over image rows can capture repeated
structure across columns and channels. For a PyTorch map `(N, C, H, W)`, select
one sample and use `features[0].permute(1, 2, 0)` before this NumPy exercise.

For `M = tensor.reshape(H, W*C)`, compute the **uncentered** SVD and truncate:

$$
M = U\,\operatorname{diag}(\sigma)\,V^T,\qquad
M_k = (U_{:, :k}\,\sigma_{:k}) V^T_{:k,:}.
$$

`U[:, :k]` is a row basis; `sigma[:k, None] * Vt[:k]` gives `k` latent scores
per column/channel pair. Reshape `M_k` back to `(H, W, C)`. It has matrix rank
**at most** `k` in this unfolding, not necessarily CP rank `k`.

The returned reconstruction is still dense and the same size as the input.
Storage compression requires retaining factors. Storing `U_k, sigma_k, Vt_k`
uses `k * (H + W*C + 1)` scalars instead of `H*W*C`, excluding metadata and at
equal precision. At high rank, the factors can cost **more** than the dense data.

An alternative is one `(H, W)` SVD per channel. That permits separate row bases
and costs `C*k*(H+W+1)` scalars. Neither unfolding is universally best; the
choice encodes which structures you expect to share.

**Predict:** if rank doubles, does retained energy double? Does a 99% energy
threshold guarantee that small image details or classifier accuracy survive?

## Setup and a reproducible image
*Preparación: construye una imagen RGB sintética sin descargar archivos.*

Run in a standard Google Colab CPU runtime. All data are generated here with a
fixed seed; no external files, pretrained model or system packages are needed.
In minimal local Jupyter, use `%pip install -q numpy torch matplotlib ipywidgets pillow`.
The synthetic RGB scene has smooth fields, sharp edges, curved detail and weak
noise. It is an image tensor used as a stand-in for a feature map, not actual
CNN activations. Real activations can be signed and need not be RGB.

In [ ]:
from __future__ import annotations

from collections.abc import Callable
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # Standard Jupyter already supports ipywidgets.

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
print(f"NumPy {np.__version__} | PyTorch {torch.__version__} | CPU")

In [ ]:
def make_image(height: int = 80, width: int = 112) -> np.ndarray:
    """Build a bounded RGB scene with detail at several spatial scales."""
    y, x = np.mgrid[0:height, 0:width]
    x, y = x / (width - 1), y / (height - 1)
    disk = ((x - 0.30)**2 + (y - 0.38)**2 < 0.15**2).astype(float)
    ridge = np.exp(-((y - 0.70 + 0.12*np.sin(8*x)) / 0.035)**2)
    checks = ((np.floor(x*14) + np.floor(y*10)) % 2) * (x > 0.60) * (y > 0.40)
    red = 0.15 + 0.40*x + 0.35*disk + 0.15*ridge
    green = 0.15 + 0.35*y + 0.25*checks + 0.20*ridge
    blue = 0.20 + 0.30*(1-x) + 0.30*disk + 0.15*np.sin(20*(x*x+y*y))
    noise = np.random.default_rng(18).normal(scale=0.012, size=(height, width, 3))
    return np.clip(np.stack([red, green, blue], axis=-1) + noise, 0, 1)


image = make_image()
print("Synthetic HWC image:", image.shape, image.dtype)

## Core activity: compress and reconstruct
*Actividad esencial: trunca la SVD y recupera los tres ejes originales.*

**Predict → Run → Explain → Check / Predice → Ejecuta → Explica → Comprueba**

Complete the starter cell below, then skip the folded
solution and run `test_compress_feature_map(compress_feature_map)` below.
Run all is also supported: it executes the reference solution and every test.

Contract: finite real numeric values in a non-empty 3D NumPy array; reject
complex, object and boolean arrays. Return floating-point values without
clipping or modifying the input. Accept integer ranks `0 <= k <= min(H, W*C)`;
rank zero returns zeros. Reject non-integer and boolean ranks. Integer images
are promoted to float64, so reconstruction does not round back to uint8.

**Hint:** `(U[:, :k] * singular_values[:k]) @ Vt[:k, :]` broadcasts along
columns of U. A full `diag` matrix is unnecessary.

In [ ]:
def compress_feature_map(tensor: np.ndarray, rank: int) -> np.ndarray:
    """Approximate an HWC tensor using rank-k SVD of its (H, W*C) unfolding."""
    # TODO: validate the input and rank; use floating-point arithmetic.
    # TODO: reshape (H, W, C) to (H, W*C).
    # TODO: np.linalg.svd(..., full_matrices=False).
    # TODO: truncate U, singular values and Vt, reconstruct, restore HWC.
    raise NotImplementedError("Complete truncated SVD")

In [ ]:
def compress_feature_map(tensor: np.ndarray, rank: int) -> np.ndarray:
    """Return a float64 HWC approximation from truncated SVD of (H, W*C).

    Rank refers to the selected matrix unfolding. No centering or clipping is
    performed. The returned dense tensor is for evaluation; store the sliced
    factors instead when actual storage reduction is the objective.
    """
    if not isinstance(tensor, np.ndarray) or tensor.ndim != 3 or 0 in tensor.shape:
        raise ValueError("tensor must have non-empty shape (H, W, C)")
    if tensor.dtype.kind not in "iuf" or not np.isfinite(tensor).all():
        raise ValueError("tensor must contain finite real numeric values")
    if isinstance(rank, (bool, np.bool_)) or not isinstance(rank, (int, np.integer)):
        raise TypeError("rank must be an integer, not a boolean")
    height, width, channels = tensor.shape
    max_rank = min(height, width * channels)
    if not 0 <= rank <= max_rank:
        raise ValueError(f"rank must be between 0 and {max_rank}")
    matrix = tensor.astype(np.float64, copy=False).reshape(height, width * channels)
    if rank == 0:
        return np.zeros(tensor.shape, dtype=np.float64)
    u, singular_values, vt = np.linalg.svd(matrix, full_matrices=False)
    reconstruction = (u[:, :rank] * singular_values[:rank]) @ vt[:rank, :]
    return reconstruction.reshape(tensor.shape)

### Energy is squared amplitude
*Energía: suma de cuadrados, no suma de valores singulares.*

$$
E(k)=\frac{\sum_{i=1}^{k}\sigma_i^2}{\sum_i\sigma_i^2},\qquad
\frac{\|M-M_k\|_F^2}{\|M\|_F^2}=1-E(k).
$$

This is the energy of the **uncentered** tensor, so a bright background may
dominate it. It is not a guarantee of perception or downstream accuracy.
The smallest rank meeting a requested threshold depends on the data; no fixed
rank guarantees 95% energy for every image. For the all-zero tensor we define
retention as 1 at all ranks, since reconstruction is already exact.

In [ ]:
def energy_curve(singular_values: np.ndarray) -> np.ndarray:
    """Return retained squared energy for ranks 0 through len(s), inclusive."""
    s = np.asarray(singular_values, dtype=float)
    if s.ndim != 1 or not np.isfinite(s).all() or np.any(s < 0) or np.any(np.diff(s) > 0):
        raise ValueError("singular values must be finite, nonnegative and descending")
    if not s.size or s[0] == 0:
        return np.ones(s.size + 1)
    # Rescaling avoids overflow when squaring a large singular value.
    squared = (s / s[0])**2
    result = np.concatenate(([0.0], np.cumsum(squared) / squared.sum()))
    result[-1] = 1.0  # Make the full-rank endpoint exact for target=1.
    return np.clip(result, 0, 1)


def rank_for_energy(singular_values: np.ndarray, target: float = 0.95) -> int:
    """Return the smallest rank retaining at least the requested energy."""
    if not np.isfinite(target) or not 0 < target <= 1:
        raise ValueError("target must lie in (0, 1]")
    return int(np.searchsorted(energy_curve(singular_values), target, side="left"))

In [ ]:
def assert_raises(error: type[Exception], operation: Callable[[], object]) -> None:
    """Assert the specified exception without requiring pytest in the runtime."""
    try:
        operation()
    except error:
        return
    raise AssertionError(f"Expected {error.__name__}")


def test_compress_feature_map(compress: Callable[[np.ndarray, int], np.ndarray]) -> None:
    """Test a known spectrum, the error identity, threshold minimality and edges."""
    # Independent oracle: singular values are exactly 5, 3, 1 and 0.
    fixture = np.diag([5., 3., 1., 0.]).reshape(4, 2, 2)
    untouched = fixture.copy()
    expected = np.diag([5., 3., 0., 0.]).reshape(fixture.shape)
    actual = compress(fixture, 2)
    assert actual.shape == fixture.shape and np.issubdtype(actual.dtype, np.floating)
    np.testing.assert_allclose(actual, expected, atol=1e-12)
    np.testing.assert_array_equal(fixture, untouched)
    spectrum = np.array([5., 3., 1., 0.])
    retention = energy_curve(spectrum)
    np.testing.assert_allclose(retention, [0, 25/35, 34/35, 1, 1])
    k95 = rank_for_energy(spectrum, 0.95)
    assert k95 == 2 and retention[k95] >= 0.95 and retention[k95-1] < 0.95
    error = np.linalg.norm(fixture - actual)**2 / np.linalg.norm(fixture)**2
    np.testing.assert_allclose(error, 1 - retention[2], atol=1e-12)
    assert error <= 0.05

    random = np.random.default_rng(180).normal(size=(7, 5, 3))
    s = np.linalg.svd(random.reshape(7, -1), compute_uv=False)
    errors = []
    for rank in range(8):
        reconstructed = compress(random, rank)
        assert reconstructed.shape == random.shape
        residual = np.linalg.norm(random - reconstructed)**2
        errors.append(residual)
        np.testing.assert_allclose(residual, np.sum(s[rank:]**2), atol=1e-10)
    assert np.all(np.diff(errors) <= 1e-10)
    np.testing.assert_allclose(compress(random, 7), random, atol=1e-12)
    # An exact low-rank tensor and a non-contiguous view preserve the contract.
    low_rank = np.outer(np.arange(1., 6.), np.arange(1., 7.)).reshape(5, 3, 2)
    np.testing.assert_allclose(compress(low_rank, 1), low_rank, atol=1e-12)
    reversed_view = random[:, ::-1, :]
    np.testing.assert_allclose(compress(reversed_view, 7), reversed_view, atol=1e-12)
    zeros = np.zeros((3, 2, 1))
    np.testing.assert_array_equal(compress(zeros, 0), zeros)
    np.testing.assert_array_equal(compress(zeros, 2), zeros)
    assert rank_for_energy(np.zeros(2)) == 0
    uint_image = np.arange(24, dtype=np.uint8).reshape(4, 3, 2)
    np.testing.assert_allclose(compress(uint_image, 4), uint_image, atol=1e-12)
    assert compress(uint_image, 1).dtype == np.float64
    for rank in (-1, 5):
        assert_raises(ValueError, lambda: compress(fixture, rank))
    for rank in (True, 1.5, "2"):
        assert_raises(TypeError, lambda: compress(fixture, rank))
    for bad in (np.zeros((3, 2)), np.zeros((0, 2, 3)), fixture + np.nan,
                fixture + np.inf, fixture.astype(complex), fixture.astype(object), fixture.astype(bool)):
        assert_raises(ValueError, lambda: compress(bad, 1))
    for target in (0, 1.1, np.nan):
        assert_raises(ValueError, lambda: rank_for_energy(spectrum, target))


test_compress_feature_map(compress_feature_map)
print("SVD: shape, known reconstruction, energy threshold and edge cases passed")

In [ ]:
# --- counterexample / contraejemplo
import numpy as np
spectrum_example = np.array([5., 3., 1.])
wrong_retention = spectrum_example[:2].sum() / spectrum_example.sum()
correct_retention = (spectrum_example[:2]**2).sum() / (spectrum_example**2).sum()
assert wrong_retention < 0.95 <= correct_retention
# --- end counterexample


## Compare ranks 1, 5, 20 and 50
*Compara la imagen, la energía retenida y el coste de guardar los factores.*

Clip only the displayed RGB values to `[0, 1]`. All errors and energy calculations
use the **unclipped** reconstruction. The rank slider caches the SVD so moving
it only reconstructs the image, rather than decomposing it again.

In [ ]:
height, width, channels = image.shape
matrix = image.reshape(height, width * channels)
u, singular_values, vt = np.linalg.svd(matrix, full_matrices=False)
retained = energy_curve(singular_values)
k95 = rank_for_energy(singular_values, 0.95)


def reconstruct_cached(rank: int) -> np.ndarray:
    """Reconstruct the demonstration image from its cached SVD factors."""
    return ((u[:, :rank] * singular_values[:rank]) @ vt[:rank]).reshape(image.shape)


def factor_storage_ratio(rank: int) -> float:
    """Factor scalar count divided by dense count, assuming equal precision."""
    return rank * (height + width*channels + 1) / image.size


ranks = (1, 5, 20, 50)
fig, axes = plt.subplots(1, 6, figsize=(18, 3.4), constrained_layout=True)
axes[0].imshow(image)
axes[0].set_title("Original")
axes[0].axis("off")
for ax, rank in zip(axes[1:5], ranks):
    reconstruction = compress_feature_map(image, rank)
    np.testing.assert_allclose(reconstruction, reconstruct_cached(rank), atol=1e-12)
    ax.imshow(np.clip(reconstruction, 0, 1))
    ax.set_title(f"Rank {rank} | E={retained[rank]:.2%}\nFactors/dense: {factor_storage_ratio(rank):.2f}×")
    ax.axis("off")
axes[5].plot(np.arange(len(retained)), retained, color="#0f766e")
axes[5].scatter(ranks, retained[list(ranks)], color="#d97706")
axes[5].axhline(0.95, color="gray", linestyle="--", label="95% target")
axes[5].axvline(k95, color="#d97706", linestyle=":", label=f"First rank: {k95}")
axes[5].set(title="Uncentered energy", xlabel="Rank k", ylabel="Retained squared energy", ylim=(0, 1.04))
axes[5].legend(fontsize=8)
plt.show()
plt.close(fig)

selected = reconstruct_cached(k95)
relative_squared_error = np.linalg.norm(image - selected)**2 / np.linalg.norm(image)**2
assert selected.shape == image.shape
assert retained[k95] >= 0.95 and (k95 == 0 or retained[k95 - 1] < 0.95)
assert relative_squared_error <= 0.05 + 1e-12
np.testing.assert_allclose(relative_squared_error, 1 - retained[k95], atol=1e-12)
print(f"First rank retaining ≥95% energy: {k95}; relative squared error: {relative_squared_error:.4f}")
print("Factors/dense > 1 means the factors use MORE scalars than the original.")

### Cross-check with PyTorch
*Verificación: compara reconstrucciones, no signos de vectores singulares.*

SVD factors may differ by sign, or rotate within tied singular subspaces.
Compare reconstructed matrices and singular values across libraries. If a
truncation cuts through tied singular values, even the optimal reconstruction
need not be unique; this seeded image has no such tie at the chosen cutoff.

In [ ]:
def test_torch_svd() -> None:
    """Verify the NumPy reconstruction against a CPU float64 PyTorch SVD."""
    tensor = torch.from_numpy(image)
    ut, st, vht = torch.linalg.svd(tensor.reshape(height, -1), full_matrices=False)
    rank = 20
    assert singular_values[rank-1] - singular_values[rank] > 1e-8
    restored = ((ut[:, :rank] * st[:rank]) @ vht[:rank]).reshape(tensor.shape)
    np.testing.assert_allclose(st.numpy(), singular_values, rtol=1e-9, atol=1e-11)
    np.testing.assert_allclose(restored.numpy(), reconstruct_cached(rank), rtol=1e-8, atol=1e-10)


test_torch_svd()
print("PyTorch SVD reconstruction matches NumPy")

## Core complete: choose a rank with evidence
*Comprueba: energía, fidelidad visual y almacenamiento son criterios distintos.*

Which rank first reaches 95% energy? At rank 50, do the three saved factors use
fewer scalars than the original? Could a small but important feature vanish
while the energy test still passes?

<details><summary>Checkpoint answer / Respuesta</summary>

Use the printed threshold rank for this image. Rank 50 stores
`50 * (80 + 112*3 + 1) = 20,850` scalars versus `26,880` dense scalars, at the
same precision. At full rank 80, factors use `33,360` scalars, which is more.
A bright background can account for most squared energy while small details
matter to recognition. Reconstruction energy alone cannot validate a CNN task.

</details>

## Explore later / Explora después

In [ ]:
def show_compression(rank: int) -> None:
    """Display the original, rank-k image, residual and selected energy."""
    reconstructed = reconstruct_cached(rank)
    fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), constrained_layout=True)
    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[1].imshow(np.clip(reconstructed, 0, 1))
    axes[1].set_title(f"Rank {rank}: {retained[rank]:.2%} energy")
    residual = np.sqrt(np.mean((image - reconstructed)**2, axis=-1))
    error_plot = axes[2].imshow(residual, vmin=0, vmax=1, cmap="magma")
    axes[2].set_title("Unclipped residual RMS")
    fig.colorbar(error_plot, ax=axes[2], fraction=0.046)
    for ax in axes[:3]:
        ax.axis("off")
    axes[3].plot(retained, color="#0f766e")
    axes[3].scatter([rank], [retained[rank]], color="#d97706")
    axes[3].set(xlabel="Rank", ylabel="Energy", ylim=(0, 1.04),
                title=f"Factors/dense: {factor_storage_ratio(rank):.2f}×")
    plt.show()
    plt.close(fig)


rank_slider = widgets.IntSlider(value=5, min=0, max=len(singular_values), description="Rank", continuous_update=False)
rank_play = widgets.Play(value=5, min=0, max=len(singular_values), interval=200)
rank_link = widgets.jslink((rank_play, "value"), (rank_slider, "value"))
compression_output = widgets.interactive_output(show_compression, {"rank": rank_slider})
display(widgets.VBox([widgets.HBox([rank_play, rank_slider]), compression_output]))
# Without a widget frontend, call show_compression(20) directly.

### How would CP differ?
*CP conserva tres factores separados; el rango matricial no es el rango CP.*

CP represents the original three axes with rank-one outer products:

$$
T_{hwc}\approx\sum_{r=1}^{R} A_{hr}B_{wr}C_{cr}.
$$

It stores `R*(H+W+C)` scalars when component weights are absorbed into a factor.
SVD of `(H, W*C)` leaves each right factor free to mix width and channels;
CP requires each component's width/channel pattern to separate into two vectors.
A CP rank-R model therefore has unfolding rank at most R, but the reverse
implication need not hold. CP fitting is iterative and generally nonconvex;
it does not inherit the matrix SVD's global optimality guarantee. The following
cell demonstrates CP reconstruction, not a CP fitting algorithm. For fitting,
continue to the workshop's CP deep dive.

In [ ]:
def reconstruct_cp(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> np.ndarray:
    """Reconstruct an HWC tensor from rank-R CP factors; weights absorbed in a."""
    if any(f.ndim != 2 for f in (a, b, c)) or not (a.shape[1] == b.shape[1] == c.shape[1]):
        raise ValueError("CP factors must be matrices with the same column count")
    return np.einsum("hr,wr,cr->hwc", a, b, c)


cp_rng = np.random.default_rng(181)
a, b, c = (cp_rng.normal(size=(size, 2)) for size in (8, 7, 3))
cp_tensor = reconstruct_cp(a, b, c)
expected_cp = sum(np.multiply.outer(np.multiply.outer(a[:, r], b[:, r]), c[:, r]) for r in range(2))
np.testing.assert_allclose(cp_tensor, expected_cp, atol=1e-12)
np.testing.assert_allclose(compress_feature_map(cp_tensor, 2), cp_tensor, atol=1e-12)
print("CP rank ≤2 → unfolding rank ≤2: reconstruction verified")

### Three short animations, with local GIF export
*Tres animaciones: expórtalas localmente sin FFmpeg ni descargas.*

The previews below load from the workshop site when online. All figures and
widgets above work independently of those URLs. Run the optional export cell
to create the same GIFs yourself; Pillow writes them without a system encoder.
Use the Play/slider explorer above to pause and inspect a chosen state.
![Reconstruct animation for module 18](https://project-delphi.github.io/tensors-workshop/images/cube-18-reconstruct.gif)

![Energy animation for module 18](https://project-delphi.github.io/tensors-workshop/images/cube-18-energy.gif)

![Terms animation for module 18](https://project-delphi.github.io/tensors-workshop/images/cube-18-terms.gif)

In [ ]:
def make_compression_animation(kind: str = "reconstruct") -> FuncAnimation:
    """Animate reconstruction, energy accumulation or individual SVD terms."""
    if kind not in ("reconstruct", "energy", "terms"):
        raise ValueError("kind must be reconstruct, energy or terms")
    original = make_image()
    h, w, c = original.shape
    left, values, right = np.linalg.svd(original.reshape(h, w*c), full_matrices=False)
    curve = np.concatenate(([0.0], np.cumsum(values**2) / np.sum(values**2)))
    ranks = (1, 2, 5, 10, 20, 50, h)
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.4), constrained_layout=True)

    def draw(frame: int) -> list:
        for ax in axes:
            ax.clear()
        rank = ranks[frame]
        restored = ((left[:, :rank] * values[:rank]) @ right[:rank]).reshape(original.shape)
        if kind == "reconstruct":
            axes[0].imshow(original)
            axes[0].set_title("Original: synthetic RGB")
            axes[1].imshow(np.clip(restored, 0, 1))
            axes[1].set_title(f"Rank {rank} | energy {curve[rank]:.2%}")
            for ax in axes:
                ax.axis("off")
        elif kind == "energy":
            axes[0].semilogy(np.arange(1, len(values)+1), values**2, color="#0f766e")
            axes[0].axvspan(.5, rank+.5, color="orange", alpha=.25)
            axes[0].set(title="Squared singular values", xlabel="Component", ylabel="Energy (log)")
            axes[1].plot(curve, color="#0f766e")
            axes[1].scatter([rank], [curve[rank]], color="#d97706")
            axes[1].set(title=f"Cumulative energy: {curve[rank]:.2%}", xlabel="Rank", ylim=(0, 1.04))
        else:
            term = (np.outer(left[:, frame], right[frame]) * values[frame]).reshape(original.shape)
            amplitude = np.abs(term).max()
            axes[0].imshow(term[..., 0], cmap="coolwarm", vmin=-amplitude, vmax=amplitude)
            axes[0].set_title(f"Signed component {frame+1}: red channel")
            cumulative = ((left[:, :frame+1] * values[:frame+1]) @ right[:frame+1]).reshape(original.shape)
            axes[1].imshow(np.clip(cumulative, 0, 1))
            axes[1].set_title(f"Sum of first {frame+1} terms: RGB")
            for ax in axes:
                ax.axis("off")
        fig.suptitle("SVD compression: patterns, reconstruction and energy")
        return []

    animation = FuncAnimation(fig, draw, frames=len(ranks), interval=800, blit=False)
    plt.close(fig)
    return animation

In [ ]:
EXPORT_GIFS = False  # Set True to write three GIF files to this runtime.
if EXPORT_GIFS:
    for kind in ('reconstruct', 'energy', 'terms'):
        animation = make_compression_animation(kind)
        destination = Path(f"cube-18-{kind}.gif")
        animation.save(str(destination), writer=PillowWriter(fps=1.25), dpi=90)
        display(destination)

### API references
*Referencias de las bibliotecas.*

[NumPy SVD](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html). For CP fitting: [Notebook 14](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/14-cp-factorization.ipynb).

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#6366f1,rgba(99,102,241,0))"></div>

## Done with this deep dive / Fin de este estudio a fondo

That is the last deep dive. The rest is back on [the workshop site](https://project-delphi.github.io/tensors-workshop/).

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Ese es el último estudio a fondo. El resto está en <a href="https://project-delphi.github.io/tensors-workshop/">el sitio del taller</a>.</div></div>

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)